# From Matplotlib Basics to Advanced Insights  
## Analyzing Player Retention & Engagement in the Gaming Industry

**A complete project covering Jake VanderPlas' Matplotlib chapter (04.00 - 04.15)**  
applied to one of the most important problems in modern games: **understanding why players stay, why they leave, and how engagement relates to monetization**.

### The Business Problem
Live-service and free-to-play games live or die by retention.  
Studios constantly ask:

- How does retention decay after install / update?
- Which player segments are most valuable and most at risk?
- When during the day/week are players most active?
- Is higher engagement correlated with higher spending?
- Are there skill or regional differences that affect long-term retention?

This notebook shows how systematic visualization with Matplotlib (and light Seaborn) turns raw telemetry into clear answers.

### Matplotlib Coverage
| Handbook Section | Technique | Gaming Application |
|------------------|-----------|--------------------|
| 04.01 | Simple Line Plots | DAU, retention curves, revenue over time |
| 04.02 | Scatter Plots | Session length vs. spend, skill vs. retention |
| 04.03 | Errorbars | Confidence on retention rates |
| 04.04 | Density & Contour | Playtime density, activity heat |
| 04.05 | Histograms & Binnings | Session length & spend distributions |
| 04.06 | Customizing Legends | Multi-series retention / cohort plots |
| 04.07 | Customizing Colorbars | Engagement heatmaps |
| 04.08 | Multiple Subplots | Executive dashboards |
| 04.09 | Text & Annotation | Call-outs on key drops or peaks |
| 04.10 | Customizing Ticks | Clean date / rank axes |
| 04.11 | Stylesheets | Consistent studio-ready style |
| 04.12 | 3D Plotting | Skill × Engagement × Spend surface |
| 04.14 | Visualization with Seaborn | Statistical overlays & categorical views |

> Goal: every plot should help a product manager, data analyst, or game designer **see what the data is trying to tell them**.


## 1. Setup & Synthetic Telemetry


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Consistent style for the whole notebook
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 10

rng = np.random.default_rng(42)

print("Libraries ready. Matplotlib style set for clean gaming analytics.")


In [ ]:
# ------------------------------------------------------------------
# Generate realistic gaming telemetry (90 days)
# ------------------------------------------------------------------
dates = pd.date_range('2025-11-01', periods=90, freq='D')
n_days = len(dates)

# Daily Active Users with weekly seasonality + soft launch growth + event spikes
base_dau = 85000 + np.linspace(0, 18000, n_days)
weekly = 9000 * np.sin(2 * np.pi * np.arange(n_days) / 7)
noise = rng.normal(0, 2500, n_days)
# Simulate two content updates / events
event = np.zeros(n_days)
event[28:35] = 22000   # mid-season event
event[63:70] = 15000   # second event
dau = np.clip(base_dau + weekly + noise + event, 60000, None).astype(int)

# Revenue (roughly follows DAU + spend intensity)
revenue = (dau * rng.uniform(0.38, 0.52, n_days) + event * 0.9).astype(int)

# Retention curves (classic decaying shape)
days_since = np.arange(0, 31)
retention_organic = 100 * np.exp(-0.085 * days_since) + 8
retention_paid    = 100 * np.exp(-0.110 * days_since) + 5
retention_event   = 100 * np.exp(-0.070 * days_since) + 11

# Player-level sample (for scatter / distributions)
n_players = 4000
player = pd.DataFrame({
    'player_id': np.arange(n_players),
    'session_min': np.clip(rng.lognormal(3.4, 0.7, n_players), 3, 240),
    'spend_usd': np.clip(rng.exponential(12, n_players), 0, 400),
    'skill_score': np.clip(rng.normal(50, 15, n_players), 5, 99),
    'days_retained': np.clip(rng.exponential(9, n_players), 1, 90).astype(int),
    'region': rng.choice(['NA', 'EU', 'ASIA', 'LATAM'], n_players, p=[0.32, 0.28, 0.27, 0.13]),
    'platform': rng.choice(['PC', 'Console', 'Mobile'], n_players, p=[0.41, 0.33, 0.26])
})
# Mild realistic relationships
player['spend_usd'] = np.clip(player['spend_usd'] + player['session_min'] * 0.07 + rng.normal(0, 4, n_players), 0, 450)
player['days_retained'] = np.clip(
    player['days_retained'] + (player['session_min'] / 15).astype(int) - (player['skill_score'] < 30) * 4,
    1, 90
)

daily = pd.DataFrame({'date': dates, 'dau': dau, 'revenue': revenue})
daily['day_of_week'] = daily['date'].dt.day_name()

print("Telemetry ready:")
print(f"  Daily series : {len(daily)} days")
print(f"  Player sample: {len(player):,} players")
display(daily.head(3))
display(player.head(3))


## 2. Simple Line Plots - Tracking the Pulse of the Game

Line plots are the most common chart in live-ops dashboards.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(daily['date'], daily['dau'], color='#2c7fb8', linewidth=2, label='DAU')
ax.set_title('Daily Active Users (90-day window)', pad=10)
ax.set_ylabel('Daily Active Users')
ax.set_xlabel('')
ax.legend()

# Nice date formatting
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("What the data tells us: Clear weekly seasonality + two distinct content-event spikes.")
print("The second event produced a smaller lift than the first -> possible event fatigue or weaker content.")


In [ ]:
# Multiple series on one axes - Retention curves
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(days_since, retention_organic, 'o-', label='Organic installs', color='#2ca25f', linewidth=2)
ax.plot(days_since, retention_paid,    's-', label='Paid UA',         color='#d95f0e', linewidth=2)
ax.plot(days_since, retention_event,   'D-', label='Event cohort',    color='#756bb1', linewidth=2)

ax.set_title('Retention Curves by Acquisition Source', pad=10)
ax.set_xlabel('Days Since Install')
ax.set_ylabel('Retention %')
ax.set_ylim(0, 105)
ax.legend(title='Cohort', frameon=True)
ax.axhline(20, color='gray', linestyle='--', alpha=0.7, label='20% benchmark')
plt.tight_layout()
plt.show()

print("Insight: Paid users drop faster than organic. Event cohort retains best -")
print("strong content is a better retention lever than pure user acquisition spend.")


## 3. Simple Scatter Plots - Relationships Between Behaviors


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Session length vs Spend
axes[0].scatter(player['session_min'], player['spend_usd'],
                alpha=0.35, s=18, c='#2c7fb8', edgecolor='none')
axes[0].set_xlabel('Avg Session Length (min)')
axes[0].set_ylabel('Lifetime Spend (USD)')
axes[0].set_title('Engagement vs Monetization')

# Skill vs Days Retained
sc = axes[1].scatter(player['skill_score'], player['days_retained'],
                     c=player['spend_usd'], cmap='viridis', alpha=0.5, s=20)
axes[1].set_xlabel('Skill Score')
axes[1].set_ylabel('Days Retained')
axes[1].set_title('Skill vs Retention (color = spend)')
cbar = fig.colorbar(sc, ax=axes[1])
cbar.set_label('Spend USD')

plt.tight_layout()
plt.show()

print("Insight: Longer sessions correlate with higher spend, but the relationship is noisy.")
print("Very low-skill players churn faster. High spenders appear across a wide skill range.")


## 4. Errorbars - Communicating Uncertainty in Retention


In [ ]:
# Simulate retention estimates with uncertainty (bootstrap-style)
cohort_days = np.array([1, 3, 7, 14, 30])
mean_ret = np.array([72, 48, 31, 19, 11])
std_ret  = np.array([3.5, 4.2, 3.8, 3.1, 2.4])   # standard error

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(cohort_days, mean_ret, yerr=std_ret,
            fmt='o-', color='#e34a33', ecolor='#fdbb84', elinewidth=2,
            capsize=5, markersize=8, label='Observed retention')
ax.set_title('Retention with 1-SE Error Bars', pad=10)
ax.set_xlabel('Day')
ax.set_ylabel('Retention %')
ax.set_xticks(cohort_days)
ax.legend()
plt.tight_layout()
plt.show()

print("Why this matters: Product teams need to know whether a 2-point retention change")
print("is real or just noise. Error bars make statistical uncertainty visible.")


## 5. Density and Contour Plots - Where Do Players Concentrate?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 2D density of session length vs skill
h = axes[0].hist2d(player['skill_score'], player['session_min'],
                   bins=30, cmap='Blues')
axes[0].set_xlabel('Skill Score')
axes[0].set_ylabel('Session Length (min)')
axes[0].set_title('Player Density: Skill × Session Length')
fig.colorbar(h[3], ax=axes[0], label='Count')

# Contour of the same relationship
from scipy.stats import gaussian_kde
xy = np.vstack([player['skill_score'], player['session_min']])
z = gaussian_kde(xy)(xy)
idx = z.argsort()
x, y, z = player['skill_score'].values[idx], player['session_min'].values[idx], z[idx]
axes[1].scatter(x, y, c=z, s=12, cmap='viridis', alpha=0.7)
axes[1].set_xlabel('Skill Score')
axes[1].set_ylabel('Session Length (min)')
axes[1].set_title('Kernel Density View')
plt.tight_layout()
plt.show()


## 6. Histograms and Binnings - Understanding Distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(player['session_min'], bins=40, color='#2c7fb8', edgecolor='white', alpha=0.85)
axes[0].axvline(player['session_min'].median(), color='red', linestyle='--', label=f"Median = {player['session_min'].median():.0f} min")
axes[0].set_title('Session Length Distribution')
axes[0].set_xlabel('Minutes')
axes[0].legend()

axes[1].hist(player['spend_usd'], bins=40, color='#d95f0e', edgecolor='white', alpha=0.85)
axes[1].set_title('Player Spend Distribution')
axes[1].set_xlabel('USD')
axes[1].set_yscale('log')   # heavy right tail typical of F2P
plt.tight_layout()
plt.show()

print("Insight: Session length is roughly log-normal. Spend is heavily skewed -")
print("a small percentage of players (whales) drive the majority of revenue.")


## 7. Customizing Legends


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for region, color in zip(['NA', 'EU', 'ASIA', 'LATAM'],
                         ['#1b9e77', '#d95f02', '#7570b3', '#e7298a']):
    subset = player[player['region'] == region]
    ax.scatter(subset['session_min'], subset['spend_usd'],
               alpha=0.4, s=20, c=color, label=region)

ax.set_xlabel('Session Length (min)')
ax.set_ylabel('Spend (USD)')
ax.set_title('Engagement vs Spend by Region')

# Custom legend
leg = ax.legend(title='Region', loc='upper right', frameon=True,
                fancybox=True, framealpha=0.95, edgecolor='gray')
leg.get_title().set_fontweight('bold')
plt.tight_layout()
plt.show()


## 8. Customizing Colorbars


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(player['skill_score'], player['days_retained'],
                c=player['session_min'], cmap='plasma', s=22, alpha=0.6)
ax.set_xlabel('Skill Score')
ax.set_ylabel('Days Retained')
ax.set_title('Retention by Skill (color = session length)')

cbar = fig.colorbar(sc, ax=ax, orientation='vertical', pad=0.02)
cbar.set_label('Avg Session (min)', rotation=270, labelpad=15)
plt.tight_layout()
plt.show()


## 9. Multiple Subplots - Building a Live-Ops Dashboard


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# DAU
axes[0, 0].plot(daily['date'], daily['dau'], color='#2c7fb8')
axes[0, 0].set_title('Daily Active Users')
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Revenue
axes[0, 1].plot(daily['date'], daily['revenue'] / 1000, color='#d95f0e')
axes[0, 1].set_title('Daily Revenue (thousands USD)')
axes[0, 1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Retention comparison
axes[1, 0].plot(days_since, retention_organic, label='Organic')
axes[1, 0].plot(days_since, retention_paid, label='Paid')
axes[1, 0].plot(days_since, retention_event, label='Event')
axes[1, 0].set_title('Retention by Cohort')
axes[1, 0].set_xlabel('Days Since Install')
axes[1, 0].legend(fontsize=9)

# Platform mix
platform_counts = player['platform'].value_counts()
axes[1, 1].bar(platform_counts.index, platform_counts.values, color=['#2c7fb8', '#d95f0e', '#31a354'])
axes[1, 1].set_title('Players by Platform')
axes[1, 1].set_ylabel('Count')

fig.autofmt_xdate()

plt.suptitle('Live-Ops Snapshot Dashboard', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 10. Text and Annotation - Highlighting What Matters


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily['date'], daily['dau'], color='#2c7fb8', linewidth=2)

# Annotate the two events
ax.annotate('Season Event\n+22k DAU',
            xy=(dates[31], dau[31]), xytext=(dates[18], dau[31] + 18000),
            arrowprops=dict(arrowstyle='->', color='darkred'),
            fontsize=10, color='darkred',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='darkred', alpha=0.9))

ax.annotate('Second Event\n(smaller lift)',
            xy=(dates[66], dau[66]), xytext=(dates[52], dau[66] + 14000),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='dimgray',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))

ax.set_title('DAU with Event Annotations')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 11. Customizing Ticks


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily['date'], daily['dau'] / 1000, color='#2c7fb8')
ax.set_ylabel('DAU (thousands)')
ax.set_title('Clean Tick Formatting')

ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}k'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 12. Stylesheets & Consistent Studio Style


In [ ]:
# Demonstrate a few built-in styles useful for reports
styles = ['seaborn-v0_8-whitegrid', 'ggplot', 'fivethirtyeight']
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

for ax, style in zip(axes, styles):
    with plt.style.context(style):
        ax.plot(days_since, retention_organic, label='Organic')
        ax.plot(days_since, retention_paid, label='Paid')
        ax.set_title(style, fontsize=10)
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("Tip: pick one style and stick to it across all internal dashboards for brand consistency.")


## 13. Three-Dimensional Plotting - Skill × Engagement × Spend


In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

sample = player.sample(800, random_state=42)
p = ax.scatter(sample['skill_score'], sample['session_min'], sample['spend_usd'],
               c=sample['days_retained'], cmap='viridis', s=18, alpha=0.7)

ax.set_xlabel('Skill Score')
ax.set_ylabel('Session (min)')
ax.set_zlabel('Spend USD')
ax.set_title('Player Landscape: Skill × Engagement × Monetization')
fig.colorbar(p, ax=ax, shrink=0.6, label='Days Retained')
plt.tight_layout()
plt.show()

print("Insight: The highest spenders tend to sit in the mid-to-high skill + high session quadrant.")
print("Very high skill + low session players spend less - they may be efficient but less engaged.")


## 14. Quick Bridge to Seaborn (Handbook 04.14)

Matplotlib gives full control; Seaborn adds statistical defaults that are often perfect for gaming KPIs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=player, x='region', y='days_retained', ax=axes[0], palette='Set2')
axes[0].set_title('Retention by Region')

sns.violinplot(data=player, x='platform', y='session_min', ax=axes[1], palette='muted')
axes[1].set_title('Session Length by Platform')
plt.tight_layout()
plt.show()


## 15. What the Data Is Telling Us - Key Insights for the Gaming Team


In [ ]:
print("=" * 60)
print("EXECUTIVE INSIGHTS FROM THE VISUAL ANALYSIS")
print("=" * 60)

print(
"1. RETENTION\n"
"   - Organic and event cohorts retain significantly better than paid UA.\n"
"   - Classic steep drop in the first 7 days - the most critical window for\n"
"     onboarding and early content hooks.\n\n"
"2. ENGAGEMENT <-> MONETIZATION\n"
"   - Session length has a positive but noisy relationship with spend.\n"
"   - A small segment of high-session players drives disproportionate revenue\n"
"     (classic F2P whale pattern visible in the log-scale spend histogram).\n\n"
"3. LIVE OPS & EVENTS\n"
"   - Content events produce clear DAU spikes.\n"
"   - The second event delivered a smaller lift -> possible diminishing returns\n"
"     or audience fatigue. Future events need stronger differentiation.\n\n"
"4. SEGMENT DIFFERENCES\n"
"   - Low-skill players churn faster.\n"
"   - Regional and platform differences exist in both retention and session length;\n"
"     these should inform localization and platform-specific roadmaps.\n\n"
"5. ACTIONABLE NEXT STEPS\n"
"   - Invest more in organic growth and high-quality events than in pure paid UA.\n"
"   - Improve early-game experience (Day 0-7) to flatten the retention curve.\n"
"   - Build targeted engagement loops for mid-skill, high-session players\n"
"     (the sweet spot of retention + spend).\n"
"   - Monitor event effectiveness with the same annotated DAU charts after every release."
)


## 16. Mapping Back to the Matplotlib Handbook

| Section | Technique Used in This Project |
|---------|--------------------------------|
| 04.01 Simple Line Plots | DAU time series, retention curves |
| 04.02 Simple Scatter Plots | Session vs Spend, Skill vs Retention |
| 04.03 Errorbars | Retention estimates with uncertainty |
| 04.04 Density & Contour | Skill × Session density |
| 04.05 Histograms | Session length & spend distributions |
| 04.06 Customizing Legends | Multi-region and multi-cohort legends |
| 04.07 Customizing Colorbars | Spend / session color mappings |
| 04.08 Multiple Subplots | Live-ops dashboard |
| 04.09 Text & Annotation | Event call-outs on DAU chart |
| 04.10 Customizing Ticks | Clean date and currency-style ticks |
| 04.11 Stylesheets | Consistent report styling |
| 04.12 Three-dimensional | Skill × Engagement × Spend landscape |
| 04.14 Seaborn | Statistical box/violin comparisons |

---

### How this type of analysis helps
Raw telemetry tables rarely change decisions.  
**Well-designed visualizations** make patterns obvious:

- Product managers see which events actually move the needle.
- UA teams see that paid users need better onboarding.
- Designers see that low-skill players are leaving too fast.
- Executives get a single dashboard view of health + monetization.

That is the real power of the Matplotlib toolkit in the gaming industry.
